# 24 The numbers behind the charts, and directions and places, on real data

The chart words of the gap list (`draw_density`, `draw_cumulative_share`, `draw_qq_dots`, `draw_smooth_line`,
`draw_error_bars`, `draw_curve`, `fit_line`, `fit_curve`, `fit_formula`, `fit_distribution`, `draw_all_dots`,
`draw_count_bars`, `sdata.draw_all_pairs`, `difference_dots`, `trade_off_curve`, `draw_richness_curves`,
`draw_threshold_check`, `draw_variogram`, `fit_dose_response`, `draw_cluster_tree`), and
`table.find_variogram_of` and `table.find_average_direction_of`.

**Only simplydata runs in this notebook.** The reference values (`references/24_charts_and_directions.csv`, from
`scripts/references/24_charts_and_directions.py`) are complete for every chart: R `lowess`, `ecdf`, `lm`,
`predict`, `nls`, `smooth.spline`, `hclust`, `stats::filter`, `tapply`, `t.test`, R circular, and formulas written
out (the Gaussian kernel density, Filliben's positions, Hurlbert's rarefaction, the mean excess, the empirical
semivariance, the exponential smoothing recursion, the Bland-Altman limits, the ROC points). Each chart's numbers are
read back with `chart.get_numbers_drawn()` (simplydata main 3fd3b85 and later): a table with one row per drawn
point (panel, layer, kind, series, x, y, low, high, value), read here with `table.get_row`. Each check takes the
same points as the reference script (the longest line, the band's low and high edge, the dots sorted, the step
heights, the bar heights) and compares them with the reference curve by their largest relative difference. The
earlier run of this notebook, which read the numbers out of the drawn figure, is described in FINDINGS.md (A16,
A17). The two table words are compared in full.

Data: R's mtcars, airquality, LakeHuron and USArrests, pROC's aSAH and vegan's BCI (written by the
reference script to `references/data/`), mcr's creatinine, ismev's rain, drc's ryegrass, the Meuse river
zinc (in `data/fetched/`) and the vanishing bearings of R circular's homing pigeons.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_rows, versions

NOTEBOOK = "24_charts_and_directions"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)
FETCHED = "../data/fetched/"


def load(name):
    """A data file the reference script wrote, read with sdata.table."""
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def drawn(chart):
    """The numbers a chart drew: chart.get_numbers_drawn(), read row by row with table.get_row."""
    return read_rows(quiet(chart.get_numbers_drawn))


def layers(points, kind=None, panel=1):
    """The drawn points of each layer of one panel (only the layers of one kind when kind= is given)."""
    found = {}
    for point in points:
        if point["panel"] == panel and (kind is None or point["kind"] == kind):
            found.setdefault(point["layer"], []).append(point)
    return list(found.values())


def longest_line(points, panel=1):
    return max(layers(points, "line", panel), key=len)


def drawn_values(points, name):
    return [point[name] for point in points]


def by_x(points):
    return sorted(points, key=lambda point: point["x"])

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## `find_variogram_of` on the Meuse zinc

The binned semivariances against gstat's bins written out (cutoff a third of the diagonal, 15 bins), and the
fitted nugget, partial sill and range against R gstat `fit.variogram` (the package simplydata calls).

In [2]:
meuse = quiet(sdata.table, "../data/fetched/meuse.csv", separator=",")
ds = "Meuse zinc (sp / gstat)"
for kind in ("spherical", "exponential"):
    found = read_rows(quiet(meuse.find_variogram_of, "zinc", x="x", y="y", of_kind=kind))
    label = f'find_variogram_of (of_kind="{kind}")'
    checks.check(ds, label, "distance of each bin", [row["distance"] for row in found])
    checks.check(ds, label, "semivariance of each bin", [row["semivariance"] for row in found])
    if "pairs" in found[0]:
        checks.check(ds, label, "pairs in each bin", [row["pairs"] for row in found])
    values = {}
    for quantity, test in [("nugget", lambda c: "nugget" in c), ("psill", lambda c: "partial sill" in c),
                           ("range", lambda c: c.startswith("fitted range") or c == "range")]:
        column = [c for c in found[0] if test(c)]
        if column:
            values[f"fitted {quantity}"] = found[0][column[0]]
    checks.check_values(ds, label, values)

agrees  Meuse zinc (sp / gstat) | find_variogram_of (of_kind="spherical") | distance of each bin: largest relative difference over 15 values: simplydata 9.99528017436e-16, gstat's bins written out, NumPy 0; difference 1.00e-15 (allowed 1e-10)
      Source: gstat's bins written out, NumPy (run reference)
agrees  Meuse zinc (sp / gstat) | find_variogram_of (of_kind="spherical") | semivariance of each bin: largest relative difference over 15 values: simplydata 0, gstat's bins written out, NumPy 0; difference 0.00e+00 (allowed 1e-10)
      Source: gstat's bins written out, NumPy (run reference)
agrees  Meuse zinc (sp / gstat) | find_variogram_of (of_kind="spherical") | pairs in each bin: largest relative difference over 15 values: simplydata 0, counted, NumPy 0; difference 0.00e+00 (allowed 1e-10)
      Source: counted, NumPy (run reference)
agrees  Meuse zinc (sp / gstat) | find_variogram_of (of_kind="spherical") | fitted nugget: simplydata 24802.6998099, R gstat fit.variogram (same packa

## `find_average_direction_of` by group: homing pigeons (R circular `pigeons`)

In [3]:
pigeons = quiet(sdata.table, data_file(NOTEBOOK, "pigeons"), separator=",")
names = pigeons.get_column_names()
print(names)
direction, group = "bearing", "treatment"
found = read_rows(quiet(pigeons.find_average_direction_of, direction, full_turn=360, for_each_group_in=group))
for line in found:
    g = line[group]
    average = [c for c in line if c.startswith("average direction")][0]
    rho = [c for c in line if "resultant length" in c][0]
    spread = [c for c in line if "circular standard deviation" in c][0]
    checks.check_values("homing pigeons (R circular pigeons)", "find_average_direction_of (for_each_group_in=)",
                        {f"average direction, {g}": line[average], f"resultant length, {g}": line[rho],
                         f"circular standard deviation, {g}": line[spread]})

['treatment', 'bearing']
agrees  homing pigeons (R circular pigeons) | find_average_direction_of (for_each_group_in=) | average direction, c: simplydata 6.31982528226, R circular mean.circular 6.31982528226; relative difference 4.22e-16 (allowed 1e-10)
      Source: R circular mean.circular (run reference) - https://search.r-project.org/CRAN/refmans/circular/html/mean.circular.html (version 0.5.2)
agrees  homing pigeons (R circular pigeons) | find_average_direction_of (for_each_group_in=) | resultant length, c: simplydata 0.745574116083, R circular rho.circular 0.745574116083; relative difference 2.98e-16 (allowed 1e-10)
      Source: R circular rho.circular (run reference) - https://search.r-project.org/CRAN/refmans/circular/html/rho.circular.html (version 0.5.2)
agrees  homing pigeons (R circular pigeons) | find_average_direction_of (for_each_group_in=) | circular standard deviation, c: simplydata 43.9052239856, R circular sd.circular 43.9052239856; relative difference 1.62e-16 (allo

## The numbers behind the charts: `chart.get_numbers_drawn()`

### Distribution charts of mtcars mpg

`draw_density` (the Gaussian kernel with Scott's bandwidth, written out), `draw_cumulative_share` (the height of
each step at the data values, against R `ecdf`), and `draw_qq_dots` (Filliben's positions; the dots and the fitted
line), for the normal and the gamma distribution.

In [4]:
mtcars = load("mtcars")
mpg = list(mtcars.get_column("mpg"))
ds = "mtcars mpg (Henderson & Velleman 1981)"

line = longest_line(drawn(sdata.chart(mtcars).draw_density(x="mpg")))
checks.check(ds, "draw_density", "density", drawn_values(line, "y"))

line = longest_line(drawn(sdata.chart(mtcars).draw_cumulative_share(x="mpg")))
step = {}
for point in line:   # the height of the step at each x (the top of the step where the line rises)
    step[point["x"]] = max(step.get(point["x"], point["y"]), point["y"])
heights = [step[x] for x in sorted(step) if x in mpg]
checks.check(ds, "draw_cumulative_share", "share at or below each value", heights)

for label, options in (("draw_qq_dots", {}), ('draw_qq_dots (distribution="gamma")', {"distribution": "gamma"})):
    points = drawn(sdata.chart(mtcars).draw_qq_dots("mpg", **options))
    dots = [p for layer in layers(points, "dot") for p in layer]
    checks.check(ds, label, "dots along (theoretical quantiles)", sorted(drawn_values(dots, "x")))
    checks.check(ds, label, "dots up (the sorted values)", sorted(drawn_values(dots, "y")))
    checks.check(ds, label, "fitted line", drawn_values(longest_line(points), "y"))

agrees  mtcars mpg (Henderson & Velleman 1981) | draw_density | density: largest relative difference over 200 values: simplydata 4.08668207615e-15, Gaussian kernel, Scott's bandwidth, NumPy 0; difference 4.09e-15 (allowed 1e-10)
      Source: Gaussian kernel, Scott's bandwidth, NumPy (run reference)
agrees  mtcars mpg (Henderson & Velleman 1981) | draw_cumulative_share | share at or below each value: largest relative difference over 25 values: simplydata 0, R ecdf 0; difference 0.00e+00 (allowed 1e-10)
      Source: R ecdf (run reference) - https://search.r-project.org/R/refmans/stats/html/ecdf.html
agrees  mtcars mpg (Henderson & Velleman 1981) | draw_qq_dots | dots along (theoretical quantiles): largest relative difference over 32 values: simplydata 0, Filliben (1975) positions 0; difference 0.00e+00 (allowed 1e-10)
      Source: Filliben (1975) positions (run reference)
agrees  mtcars mpg (Henderson & Velleman 1981) | draw_qq_dots | dots up (the sorted values): largest relative diff

### Smooth lines

`draw_smooth_line` on mtcars weight and mpg: LOWESS (against R `lowess` with delta 0 and with its default delta,
the mean of the line at each weight), the smoothing spline with `smoothness=0.01` (R `smooth.spline(lambda = 0.01)`,
also on the Lake Huron years, where no x repeats), loess and its confidence band (R `loess` and
`predict(se = TRUE)`). On the Lake Huron levels: the moving averages (R `stats::filter`, inside the ends),
Savitzky-Golay (SciPy `savgol_filter`, the parent) and exponential smoothing (the recursion written out).

In [5]:
ds = "mtcars weight and mpg"
line = longest_line(drawn(sdata.chart(mtcars).draw_dots(x="wt", y="mpg").draw_smooth_line()))
at_x = {}
for point in line:
    at_x.setdefault(point["x"], []).append(point["y"])
checks.check(ds, "draw_smooth_line", "LOWESS line", [sum(at_x[x]) / len(at_x[x]) for x in sorted(at_x)])

huron = load("huron")
for name, frame, x, y in (("mtcars weight and mpg", mtcars, "wt", "mpg"), ("Lake Huron levels (no ties)", huron, "year", "level")):
    line = longest_line(drawn(sdata.chart(frame).draw_dots(x=x, y=y).draw_smooth_line(test_method="spline", smoothness=0.01)))
    checks.check(name, 'draw_smooth_line (test_method="spline", smoothness=0.01)', "spline", drawn_values(line, "y"))

points = drawn(sdata.chart(mtcars).draw_dots(x="wt", y="mpg").draw_smooth_line(test_method="loess", with_confidence_band=True))
checks.check(ds, 'draw_smooth_line (test_method="loess")', "loess line", drawn_values(longest_line(points), "y"))
band = by_x(layers(points, "band")[0])
checks.check(ds, "draw_smooth_line (with_confidence_band=True)", "band low", drawn_values(band, "low"))
checks.check(ds, "draw_smooth_line (with_confidence_band=True)", "band high", drawn_values(band, "high"))

ds = "Lake Huron levels 1875-1972 (Brockwell & Davis)"
levels = list(huron.get_column("level"))


def smooth_line(**options):
    """The smoothed line: the last line that is not the data line itself."""
    lines = layers(drawn(sdata.chart(huron).draw_lines(x="year", y="level").draw_smooth_line(**options)), "line")
    smooth = [line for line in lines if drawn_values(line, "y") != levels]
    return smooth[-1] if smooth else max(lines, key=len)


for options in ({"test_method": "moving average", "window": 5}, {"test_method": "moving average", "weights": [1, 2, 3, 2, 1]}):
    label = "draw_smooth_line (" + ", ".join(f"{k}={v!r}" for k, v in options.items()) + ")"
    inside = [y for y in drawn_values(smooth_line(**options), "y") if y is not None]
    checks.check(ds, label, "moving average (inside the ends)", inside)
checks.check(ds, 'draw_smooth_line (test_method="savitzky-golay", of_degree=2)', "smoothed line",
             drawn_values(smooth_line(test_method="savitzky-golay", of_degree=2), "y"))
checks.check(ds, 'draw_smooth_line (test_method="exponential", newest_weight=0.3)', "smoothed line",
             drawn_values(smooth_line(test_method="exponential", newest_weight=0.3), "y"))

explained difference – reference (C28)  mtcars weight and mpg | draw_smooth_line | LOWESS line: largest relative difference over 29 values: simplydata 2.9323025901e-05, R lowess(delta = 0) 0; difference 2.93e-05 (allowed 1e-06)
      why: R lowess(delta = 0) is another line than simplydata's, now that simplydata uses R's default delta - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c28-r-lowessdelta--0-is-another-line-than-simplydatas-now-that-simplydata-uses-rs-default-delta
      Source: R lowess(delta = 0) (run reference) - https://search.r-project.org/R/refmans/stats/html/lowess.html
agrees  mtcars weight and mpg | draw_smooth_line | LOWESS line: largest relative difference over 29 values: simplydata 1.46285891354e-10, R lowess (its default delta, 1 % of the range) 0; difference 1.46e-10 (allowed 1e-06)
      Source: R lowess (its default delta, 1 % of the range) (run reference) - https://search.r-project.org/R/refmans/stats/html/lowess.html
explained diff

agrees  Lake Huron levels (no ties) | draw_smooth_line (test_method="spline", smoothness=0.01) | spline: largest relative difference over 200 values: simplydata 2.03802792673e-08, R smooth.spline(lambda = 0.01) 0; difference 2.04e-08 (allowed 1e-06)
      Source: R smooth.spline(lambda = 0.01) (run reference) - https://search.r-project.org/R/refmans/stats/html/smooth.spline.html


agrees  mtcars weight and mpg | draw_smooth_line (test_method="loess") | loess line: largest relative difference over 200 values: simplydata 0, R loess (same package) 0; difference 0.00e+00 (allowed 1e-10)
      Source: R loess (same package) (run reference) - https://search.r-project.org/R/refmans/stats/html/loess.html
agrees  mtcars weight and mpg | draw_smooth_line (with_confidence_band=True) | band low: largest relative difference over 200 values: simplydata 0, R predict(loess, se = TRUE) (same package) 0; difference 0.00e+00 (allowed 1e-10)
      Source: R predict(loess, se = TRUE) (same package) (run reference) - https://search.r-project.org/R/refmans/stats/html/predict.html
agrees  mtcars weight and mpg | draw_smooth_line (with_confidence_band=True) | band high: largest relative difference over 200 values: simplydata 0, R predict(loess, se = TRUE) (same package) 0; difference 0.00e+00 (allowed 1e-10)
      Source: R predict(loess, se = TRUE) (same package) (run reference) - http

agrees  Lake Huron levels 1875-1972 (Brockwell & Davis) | draw_smooth_line (test_method='moving average', window=5) | moving average (inside the ends): largest relative difference over 94 values: simplydata 1.97114953206e-16, R stats::filter(sides = 2) 0; difference 1.97e-16 (allowed 1e-10)
      Source: R stats::filter(sides = 2) (run reference)
agrees  Lake Huron levels 1875-1972 (Brockwell & Davis) | draw_smooth_line (test_method='moving average', weights=[1, 2, 3, 2, 1]) | moving average (inside the ends): largest relative difference over 94 values: simplydata 3.92603458171e-16, R stats::filter(sides = 2) 0; difference 3.93e-16 (allowed 1e-10)
      Source: R stats::filter(sides = 2) (run reference)


agrees  Lake Huron levels 1875-1972 (Brockwell & Davis) | draw_smooth_line (test_method="savitzky-golay", of_degree=2) | smoothed line: largest relative difference over 98 values: simplydata 0, SciPy savgol_filter (window 5, the parent) 0; difference 0.00e+00 (allowed 1e-10)
      Source: SciPy savgol_filter (window 5, the parent) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.savgol_filter.html
agrees  Lake Huron levels 1875-1972 (Brockwell & Davis) | draw_smooth_line (test_method="exponential", newest_weight=0.3) | smoothed line: largest relative difference over 98 values: simplydata 0, the recursion written out, NumPy 0; difference 0.00e+00 (allowed 1e-10)
      Source: the recursion written out, NumPy (run reference)


1

### Error bars, curves and fitted lines

`draw_error_bars` of mpg by cylinders (R `tapply` and `t.test`); `draw_curve` of a formula and of a distribution
(R `dnorm`, `pnorm`); `fit_line` with its confidence band (R `lm`, `predict(interval = "confidence")`); `fit_curve`
(R `lm(poly(raw = TRUE))`); `fit_formula` (R `nls` and SciPy `curve_fit`); `fit_distribution` of the airquality ozone
(R `dgamma` at the fitdistrplus fit, and the gamma density at the maximum likelihood fit).

In [6]:
ds = "mtcars mpg by cylinders"
for kind in ("standard deviation", "standard error", "confidence interval"):
    points = drawn(sdata.chart(mtcars).draw_dots(x="cyl", y="mpg").draw_error_bars(kind))
    bars = by_x([p for layer in layers(points, "error bar") for p in layer])
    label = f'draw_error_bars (of_kind="{kind}")'
    checks.check(ds, label, "bar bottoms", drawn_values(bars, "low"))
    checks.check(ds, label, "bar tops", drawn_values(bars, "high"))

ds = "mtcars weight and mpg"
line = longest_line(drawn(sdata.chart(mtcars).draw_dots(x="wt", y="mpg").draw_curve("y = 40 - 5 * x", between=[1, 6], steps=50)))
checks.check(ds, "draw_curve (steps=50)", "points drawn", len(line))
checks.check(ds, "draw_curve (formula_or_distribution=a formula)", "curve", drawn_values(line, "y"))
normal = sdata.distribution("normal", average=20, standard_deviation=6)
for show in ("density", "cumulative"):
    line = longest_line(drawn(sdata.chart(mtcars).draw_curve(normal, between=[5, 35], show=show)))
    checks.check(ds, f'draw_curve (a distribution, show="{show}")', show, drawn_values(line, "y"))

points = drawn(sdata.chart(mtcars).draw_dots(x="wt", y="mpg").fit_line(with_confidence_band=True))
checks.check(ds, "fit_line", "line", drawn_values(longest_line(points), "y"))
band = by_x(layers(points, "band")[0])
checks.check(ds, "fit_line (with_confidence_band=True)", "band low", drawn_values(band, "low"))
checks.check(ds, "fit_line (with_confidence_band=True)", "band high", drawn_values(band, "high"))
for degree in (2, 3):
    line = longest_line(drawn(sdata.chart(mtcars).draw_dots(x="wt", y="mpg").fit_curve(of_degree=degree)))
    checks.check(ds, f"fit_curve (of_degree={degree})", "curve", drawn_values(line, "y"))
line = longest_line(drawn(sdata.chart(mtcars).draw_dots(x="wt", y="mpg").fit_formula("mpg = a * exp(b * wt)", start={"a": 40, "b": -0.3})))
checks.check(ds, "fit_formula (formula=, start=)", "curve", drawn_values(line, "y"))

airquality = load("airquality")
line = longest_line(drawn(sdata.chart(airquality).draw_histogram(x="Ozone", show="density").fit_distribution("gamma")))
checks.check("airquality ozone", 'fit_distribution (distribution="gamma")', "curve", drawn_values(line, "y"))

Error bars: the standard deviation of the values (how spread out they are around the average).


agrees  mtcars mpg by cylinders | draw_error_bars (of_kind="standard deviation") | bar bottoms: largest relative difference over 3 values: simplydata 1.94251043044e-16, R tapply / t.test 0; difference 1.94e-16 (allowed 1e-10)
      Source: R tapply / t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
agrees  mtcars mpg by cylinders | draw_error_bars (of_kind="standard deviation") | bar tops: largest relative difference over 3 values: simplydata 1.67609104629e-16, R tapply / t.test 0; difference 1.68e-16 (allowed 1e-10)
      Source: R tapply / t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
Error bars: the standard error of the average (how sure we are about the average itself).
agrees  mtcars mpg by cylinders | draw_error_bars (of_kind="standard error") | bar bottoms: largest relative difference over 3 values: simplydata 1.85100216269e-16, R tapply / t.test 0; difference 1.85e-16 (allowed 1e-10)
      Source: R

agrees  mtcars mpg by cylinders | draw_error_bars (of_kind="confidence interval") | bar bottoms: largest relative difference over 3 values: simplydata 5.79293017844e-16, R tapply / t.test 0; difference 5.79e-16 (allowed 1e-10)
      Source: R tapply / t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
agrees  mtcars mpg by cylinders | draw_error_bars (of_kind="confidence interval") | bar tops: largest relative difference over 3 values: simplydata 2.14301260785e-16, R tapply / t.test 0; difference 2.14e-16 (allowed 1e-10)
      Source: R tapply / t.test (run reference) - https://search.r-project.org/R/refmans/stats/html/tapply.html
agrees  mtcars weight and mpg | draw_curve (steps=50) | points drawn: simplydata 50, steps= 50; difference 0.00e+00 (allowed 0e+00)
      Source: steps= (run reference)
agrees  mtcars weight and mpg | draw_curve (formula_or_distribution=a formula) | curve: largest relative difference over 50 values: simplydata 0, the formul

agrees  mtcars weight and mpg | fit_line | line: largest relative difference over 200 values: simplydata 1.71282962503e-15, R lm 0; difference 1.71e-15 (allowed 1e-10)
      Source: R lm (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  mtcars weight and mpg | fit_line (with_confidence_band=True) | band low: largest relative difference over 200 values: simplydata 4.99171157102e-15, R predict(interval = "confidence") 0; difference 4.99e-15 (allowed 1e-10)
      Source: R predict(interval = "confidence") (run reference) - https://search.r-project.org/R/refmans/stats/html/predict.html
agrees  mtcars weight and mpg | fit_line (with_confidence_band=True) | band high: largest relative difference over 200 values: simplydata 2.37301524237e-15, R predict(interval = "confidence") 0; difference 2.37e-15 (allowed 1e-10)
      Source: R predict(interval = "confidence") (run reference) - https://search.r-project.org/R/refmans/stats/html/predict.html


agrees  mtcars weight and mpg | fit_curve (of_degree=2) | curve: largest relative difference over 200 values: simplydata 1.82011112997e-15, R lm(poly(raw = TRUE)) 0; difference 1.82e-15 (allowed 1e-10)


      Source: R lm(poly(raw = TRUE)) (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html
agrees  mtcars weight and mpg | fit_curve (of_degree=3) | curve: largest relative difference over 200 values: simplydata 2.3929092905e-15, R lm(poly(raw = TRUE)) 0; difference 2.39e-15 (allowed 1e-10)
      Source: R lm(poly(raw = TRUE)) (run reference) - https://search.r-project.org/R/refmans/stats/html/lm.html


agrees  mtcars weight and mpg | fit_formula (formula=, start=) | curve: largest relative difference over 200 values: simplydata 1.20517030225e-07, R nls (its default stopping rule) 0; difference 1.21e-07 (allowed 1e-06)
      Source: R nls (its default stopping rule) (run reference) - https://search.r-project.org/R/refmans/stats/html/nls.html
agrees  mtcars weight and mpg | fit_formula (formula=, start=) | curve: largest relative difference over 200 values: simplydata 1.0629929492e-08, SciPy curve_fit (tight) 0; difference 1.06e-08 (allowed 1e-06)
      Source: SciPy curve_fit (tight) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.curve_fit.html
explained difference – reference (C10)  airquality ozone | fit_distribution (distribution="gamma") | curve: largest relative difference over 200 values: simplydata 1.42849929412e-06, R dgamma at fitdistrplus fit (reltol 1e-14) 0; difference 1.43e-06 (allowed 1e-06)
      why: notebook 23: compare_distr

2

### Dots, bars and pairs

`draw_all_dots` and `draw_count_bars` (R `table`) by cylinders, and `sdata.draw_all_pairs` of mpg and weight: each
panel of dots must hold the two columns' values (1 = yes).

In [7]:
ds = "mtcars"
points = drawn(sdata.chart(mtcars).draw_all_dots(x="cyl", y="mpg"))
checks.check(ds, "draw_all_dots", "heights of the dots, sorted", sorted(p["y"] for layer in layers(points, "dot") for p in layer))
points = drawn(sdata.chart(mtcars).draw_count_bars(x="cyl"))
checks.check(ds, "draw_count_bars", "bar heights, sorted", sorted(p["y"] for layer in layers(points, "bar") for p in layer))

points = drawn(quiet(sdata.draw_all_pairs, mtcars, columns=["mpg", "wt"]))
column = {"mpg": sorted(mpg), "wt": sorted(mtcars.get_column("wt"))}


def same(a, b):
    return len(a) == len(b) and all(abs(u - v) <= 1e-12 * max(abs(v), 1.0) for u, v in zip(a, b))


panels = sorted({p["panel"] for p in points if p["kind"] == "dot"})
for k, (panel, (across, up)) in enumerate(zip(panels, [("wt", "mpg"), ("mpg", "wt")])):
    dots = [p for layer in layers(points, "dot", panel) for p in layer]
    ok = same(sorted(drawn_values(dots, "x")), column[across]) and same(sorted(drawn_values(dots, "y")), column[up])
    checks.check(ds, "sdata.draw_all_pairs", f"panel {k + 1}: the dots are the two columns' values", 1.0 if ok else 0.0)

agrees  mtcars | draw_all_dots | heights of the dots, sorted: largest relative difference over 32 values: simplydata 0, the values, sorted 0; difference 0.00e+00 (allowed 1e-10)
      Source: the values, sorted (run reference)
agrees  mtcars | draw_count_bars | bar heights, sorted: largest relative difference over 3 values: simplydata 0, R table 0; difference 0.00e+00 (allowed 1e-10)
      Source: R table (run reference) - https://search.r-project.org/R/refmans/stats/html/table.html


agrees  mtcars | sdata.draw_all_pairs | panel 1: the dots are the two columns' values: simplydata 1, the data 1; difference 0.00e+00 (allowed 0e+00)
      Source: the data (run reference)
agrees  mtcars | sdata.draw_all_pairs | panel 2: the dots are the two columns' values: simplydata 1, the data 1; difference 0.00e+00 (allowed 0e+00)
      Source: the data (run reference)


### Agreement and diagnostic charts

`difference_dots` of mcr's creatinine (serum against plasma; the dots and the three lines of Bland & Altman 1986)
and `trade_off_curve` of pROC's aSAH: the drawn points against the (false alarms, found) pair of every threshold,
the AUC in the legend (the series name of the curve) against the Mann-Whitney AUC, and the marked best cut-off's
point and the cut-off written next to it (the value column of the marked dot, simplydata a5a8ff0 and later) against
Youden's index and R pROC `coords(..., "best")`.

In [8]:
creatinine = quiet(sdata.table, FETCHED + "creatinine.csv", separator=",")
ds = "creatinine, serum and plasma (R mcr)"
points = drawn(sdata.chart(creatinine).difference_dots(first="serum", second="plasma"))
dots = sorted([p for layer in layers(points, "dot") for p in layer], key=lambda p: (p["x"], p["y"]))
checks.check(ds, "difference_dots", "dots along (averages)", drawn_values(dots, "x"))
checks.check(ds, "difference_dots", "dots up (differences)", drawn_values(dots, "y"))
flat = [line for line in layers(points, "line") if len(line) == 2 and line[0]["y"] == line[1]["y"]]
checks.check(ds, "difference_dots", "the three lines (lower limit, average, upper limit)",
             sorted({round(line[0]["y"], 12) for line in flat}))

asah = load("asah")
ds = "aSAH (R pROC example; Turck et al. 2010)"
points = drawn(sdata.chart(asah).trade_off_curve(score="s100b", truth="outcome"))
curve = longest_line(points)
pairs = sorted({(round(p["x"], 12), round(p["y"], 12)) for p in curve})
checks.check(ds, "trade_off_curve", "the drawn points without repeats, in order: false alarms", [a for a, _ in pairs])
checks.check(ds, "trade_off_curve", "the drawn points without repeats, in order: found", [b for _, b in pairs])
legend = curve[0]["series"]
print("the curve's name in the legend:", legend)
checks.check(ds, "trade_off_curve", "AUC in the legend (printed)", float(legend.split("AUC ")[1].split(" ")[0]))
marked = [p for layer in layers(points, "dot") for p in layer if p["series"] == legend]
checks.check(ds, "trade_off_curve", "best cut-off marked: its point (false alarms, found)", [marked[0]["x"], marked[0]["y"]])
checks.check(ds, "trade_off_curve", "best cut-off marked", marked[0]["value"])

agrees  creatinine, serum and plasma (R mcr) | difference_dots | dots along (averages): largest relative difference over 108 values: simplydata 0, (first + second) / 2 0; difference 0.00e+00 (allowed 1e-10)
      Source: (first + second) / 2 (run reference)
agrees  creatinine, serum and plasma (R mcr) | difference_dots | dots up (differences): largest relative difference over 108 values: simplydata 0, first - second 0; difference 0.00e+00 (allowed 1e-10)
      Source: first - second (run reference)
agrees  creatinine, serum and plasma (R mcr) | difference_dots | the three lines (lower limit, average, upper limit): largest relative difference over 3 values: simplydata 2.40954846437e-11, Bland & Altman (1986), NumPy 0; difference 2.41e-11 (allowed 1e-10)
      Source: Bland & Altman (1986), NumPy (run reference)
agrees  aSAH (R pROC example; Turck et al. 2010) | trade_off_curve | the drawn points without repeats, in order: false alarms: largest relative difference over 51 values: simplyd

2

### Ecology, extremes, variograms, dose-response and cluster trees

`draw_richness_curves` of BCI plots 1 to 5 (Hurlbert's rarefaction; each curve is matched to its plot by the
plot's total count, where the curve ends); `draw_threshold_check` of the Coles rainfall (the mean excess and its
95 % interval, the top panel); `draw_variogram` of the Meuse zinc (the binned semivariances); `fit_dose_response`
of drc's ryegrass (SciPy `curve_fit`, above concentration 0); and the join heights of `draw_cluster_tree`
(R `hclust`), each join drawn as a bracket of four points whose second point is at the join's height.

In [9]:
plots = load("bci_plots_1_to_5")
species = [c for c in plots.get_column_names() if c != "plot"]
totals = [sum(row[s] for s in species) for row in read_rows(plots)]
ds = "Barro Colorado Island, plots 1-5 (vegan BCI)"
curves = sorted(layers(drawn(sdata.chart(plots).draw_richness_curves(columns=species, names="plot")), "line"), key=len, reverse=True)[:5]
for curve in curves:
    end = max(drawn_values(curve, "x"))
    which = min(range(len(totals)), key=lambda i: abs(totals[i] - end))
    checks.check(ds, "draw_richness_curves", f"curve of plot {which + 1}", drawn_values(curve, "y"))

rain = quiet(sdata.table, FETCHED + "coles_rain.csv", separator=",")
ds = "daily rainfall, south-west England (Coles 2001)"
points = drawn(sdata.chart(rain).draw_threshold_check(x="rain", between=[10, 30]))
lines = sorted([line for line in layers(points, "line", panel=1) if len(line) > 2],
               key=lambda line: sum(drawn_values(line, "y")) / len(line))
checks.check(ds, "draw_threshold_check", "mean excess", drawn_values(lines[1], "y"))
checks.check(ds, "draw_threshold_check", "mean excess, 95 % interval low", drawn_values(lines[0], "y"))
checks.check(ds, "draw_threshold_check", "mean excess, 95 % interval high", drawn_values(lines[2], "y"))

meuse = quiet(sdata.table, FETCHED + "meuse.csv", separator=",")
ds = "Meuse zinc (sp / gstat)"
for kind in ("spherical", "exponential"):
    points = drawn(sdata.chart(meuse).draw_variogram("zinc", x="x", y="y", of_kind=kind))
    dots = by_x([p for layer in layers(points, "dot") for p in layer])
    checks.check(ds, f'draw_variogram (of_kind="{kind}")', "dots up (semivariances)", drawn_values(dots, "y"))

rye = quiet(sdata.table, FETCHED + "ryegrass.csv", separator=",")
line = longest_line(drawn(sdata.chart(rye).draw_dots(x="conc", y="root_length").fit_dose_response()))
checks.check("ryegrass (drc; Inderjit et al. 2002)", "fit_dose_response", "curve", [p["y"] for p in line if p["x"] > 0])


def join_heights(points):
    """The height of each join: the second point of each bracket of four points, smallest first."""
    return sorted(line[1]["y"] for line in layers(points, "line") if len(line) == 4)


usarrests = load("usarrests")
points = drawn(sdata.chart(usarrests).draw_cluster_tree(columns=["Murder", "Assault", "UrbanPop", "Rape"], names="row name"))
checks.check("USArrests (McNeil 1977)", "draw_cluster_tree", "join heights (Ward, scaled columns)", join_heights(points))
sites = load("bci_plots_1_to_12")
for linkage, method in ((None, "average"), ("complete", "complete")):
    options = {"linkage": linkage} if linkage else {}
    points = drawn(sdata.chart(sites).draw_cluster_tree(columns=species, measure="bray-curtis", names="plot", **options))
    label = 'draw_cluster_tree (measure="bray-curtis"' + (f', linkage="{linkage}")' if linkage else ")")
    checks.check("Barro Colorado Island, plots 1-12 (vegan BCI)", label, f"join heights ({method})", join_heights(points))

Richness curves of 5 rows from 225 columns: the expected number of kinds present at each total count (R vegan::rarecurve, the expected values of vegan::rarefy). The dotted line marks the smallest total, 435: compare richness at or below it (table.add_diversity_columns(at_total_count=435)). Curves that still climb at their end would find more kinds with more counts.
agrees  Barro Colorado Island, plots 1-5 (vegan BCI) | draw_richness_curves | curve of plot 4: largest relative difference over 65 values: simplydata 1.6779355683e-11, Hurlbert (1971), NumPy 0; difference 1.68e-11 (allowed 1e-10)
      Source: Hurlbert (1971), NumPy (run reference)
agrees  Barro Colorado Island, plots 1-5 (vegan BCI) | draw_richness_curves | curve of plot 5: largest relative difference over 64 values: simplydata 1.3968604051e-11, Hurlbert (1971), NumPy 0; difference 1.40e-11 (allowed 1e-10)
      Source: Hurlbert (1971), NumPy (run reference)
agrees  Barro Colorado Island, plots 1-5 (vegan BCI) | draw_richne

agrees  daily rainfall, south-west England (Coles 2001) | draw_threshold_check | mean excess: largest relative difference over 23 values: simplydata 2.25620757938e-16, mean of x - u above u, NumPy 0; difference 2.26e-16 (allowed 1e-10)
      Source: mean of x - u above u, NumPy (run reference)
agrees  daily rainfall, south-west England (Coles 2001) | draw_threshold_check | mean excess, 95 % interval low: largest relative difference over 23 values: simplydata 2.44421128261e-16, mean - 1.96 sd / sqrt(n), NumPy 0; difference 2.44e-16 (allowed 1e-10)
      Source: mean - 1.96 sd / sqrt(n), NumPy (run reference)
agrees  daily rainfall, south-west England (Coles 2001) | draw_threshold_check | mean excess, 95 % interval high: largest relative difference over 23 values: simplydata 2.14192922017e-16, mean + 1.96 sd / sqrt(n), NumPy 0; difference 2.14e-16 (allowed 1e-10)
      Source: mean + 1.96 sd / sqrt(n), NumPy (run reference)
Variogram of "zinc" (155 places): each dot is a bin of pairs of 

agrees  ryegrass (drc; Inderjit et al. 2002) | fit_dose_response | curve: largest relative difference over 300 values: simplydata 6.43180197924e-07, SciPy curve_fit (tight) 0; difference 6.43e-07 (allowed 1e-06)
      Source: SciPy curve_fit (tight) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.curve_fit.html
Cluster tree of 50 rows from 4 columns: Euclidean distances (each column on the same scale first), ward linkage. Cophenetic correlation 0.70: how well the heights of the tree keep the distances (1 = exactly).
simplydata chooses linkage="ward" for Euclidean distances (the same as table.add_clusters).
agrees  USArrests (McNeil 1977) | draw_cluster_tree | join heights (Ward, scaled columns): largest relative difference over 49 values: simplydata 1.0786516609e-15, R hclust(method = "ward.D2") on scaled columns 0; difference 1.08e-15 (allowed 1e-10)
      Source: R hclust(method = "ward.D2") on scaled columns (run reference) - https://search.

## Result

In [10]:
checks.finish()

24_charts_and_directions: 86 comparisons - 83 agrees, 3 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,24_charts_and_directions,Meuse zinc (sp / gstat),"find_variogram_of (of_kind=""spherical"")",distance of each bin: largest relative differe...,9.995280e-16,0.000000,"gstat's bins written out, NumPy",9.995280e-16,1.000000e-10,absolute,9.995280e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,24_charts_and_directions,Meuse zinc (sp / gstat),"find_variogram_of (of_kind=""spherical"")",semivariance of each bin: largest relative dif...,0.000000e+00,0.000000,"gstat's bins written out, NumPy",0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,24_charts_and_directions,Meuse zinc (sp / gstat),"find_variogram_of (of_kind=""spherical"")",pairs in each bin: largest relative difference...,0.000000e+00,0.000000,"counted, NumPy",0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,24_charts_and_directions,Meuse zinc (sp / gstat),"find_variogram_of (of_kind=""spherical"")",fitted nugget,2.480270e+04,24802.699810,R gstat fit.variogram (same package),0.000000e+00,1.000000e-06,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/gsta...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,24_charts_and_directions,Meuse zinc (sp / gstat),"find_variogram_of (of_kind=""spherical"")",fitted psill,1.347465e+05,134746.513226,R gstat fit.variogram (same package),0.000000e+00,1.000000e-06,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/gsta...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,24_charts_and_directions,Meuse zinc (sp / gstat),"draw_variogram (of_kind=""exponential"")",dots up (semivariances): largest relative diff...,0.000000e+00,0.000000,"gstat's bins written out, NumPy",0.000000e+00,1.000000e-10,absolute,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
82,24_charts_and_directions,ryegrass (drc; Inderjit et al. 2002),fit_dose_response,curve: largest relative difference over 300 va...,6.431802e-07,0.000000,SciPy curve_fit (tight),6.431802e-07,1.000000e-06,absolute,6.431802e-07,PASS,,agrees,,,,run reference,https://docs.scipy.org/doc/scipy-1.18.1/refere...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
83,24_charts_and_directions,USArrests (McNeil 1977),draw_cluster_tree,"join heights (Ward, scaled columns): largest r...",1.078652e-15,0.000000,"R hclust(method = ""ward.D2"") on scaled columns",1.078652e-15,1.000000e-10,absolute,1.078652e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
84,24_charts_and_directions,"Barro Colorado Island, plots 1-12 (vegan BCI)","draw_cluster_tree (measure=""bray-curtis"")",join heights (average): largest relative diffe...,2.945320e-16,0.000000,"R hclust(""average"") on vegan vegdist(""bray"")",2.945320e-16,1.000000e-10,absolute,2.945320e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
